# 27 · Load: ACS PUMS 2020–2024, NYC residents → `staging`

**Source card:** `acs_pums.md` (§6 issues, §7 staging table). Explored in: notebook 10. Decisions: D4 (field of degree → CIP, `ref/ref_fod_cip.csv`), D5 (young-worker share as a junior/senior signal).

| Table | One row = | Expected rows |
|---|---|---|
| `stg_census__acs_pums_person` | one NYC resident aged 16+ in the 2020–24 pooled sample | 279,311 |

Person-level rows stay in staging only; core keeps weighted aggregates with margins of error from the 80 replicate weights, and suppresses cells under 30 rows (brief S13). Pay is converted to 2024 dollars with `ADJINC`. Streaming the 640 MB state file takes about a minute.

In [1]:
import sys
import zipfile
from pathlib import Path

import pandas as pd
from IPython.display import display

PROJECT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "common.py").exists())
sys.path.insert(0, str(PROJECT / "src"))
from common import log
from staging import connect, raw_files, stage

pd.set_option("display.max_columns", 40, "display.max_colwidth", 70, "display.width", 220)
NB = "27-load-acs-pums.ipynb"
BOROUGHS = {"005": "Bronx", "047": "Brooklyn", "061": "Manhattan", "081": "Queens", "085": "Staten Island"}
NYC_POW = {"04100", "04200", "04300", "04400", "04500"}
conn = connect()
F = raw_files("acs_pums")
P = {Path(f).name: p for f, p in zip(F.file, F.path)}

## 1. NYC = 55 PUMAs (2020 definitions)

A PUMA never crosses a borough line, so borough is exact.

In [2]:
tr = pd.read_csv(P["2020_Census_Tract_to_2020_PUMA.txt"], dtype=str, encoding="utf-8-sig")
nyc_tr = tr[(tr.STATEFP == "36") & tr.COUNTYFP.isin(BOROUGHS)]
assert nyc_tr.groupby("PUMA5CE").COUNTYFP.nunique().max() == 1
PUMA_BORO = {p: BOROUGHS[c] for p, c in zip(nyc_tr.PUMA5CE, nyc_tr.COUNTYFP)}
print(f"{len(PUMA_BORO)} NYC PUMAs"); assert len(PUMA_BORO) == 55

55 NYC PUMAs


## 2. Stream the NY State person file, keep NYC residents

In [3]:
REP = [f"PWGTP{i}" for i in range(1, 81)]
USE = ["SERIALNO", "SPORDER", "PUMA", "ADJINC", "PWGTP", "AGEP", "SEX", "SCHL", "SCHG", "FOD1P", "FOD2P", "ESR", "COW",
       "OCCP", "SOCP", "NAICSP", "WAGP", "PERNP", "WKHP", "WKWN", "POWSP", "POWPUMA", "FFODP", "FOCCP", "FWAGP"] + REP
zf = zipfile.ZipFile(P["csv_pny_2020-2024_5yr.zip"])
n_state, keep = 0, []
for ch in pd.read_csv(zf.open("psam_p36.csv"), dtype=str, keep_default_na=False, usecols=USE, chunksize=400_000):
    n_state += len(ch)
    keep.append(ch[ch.PUMA.isin(PUMA_BORO)])
r = pd.concat(keep, ignore_index=True).apply(lambda s: s.str.strip())
log("NY State persons -> NYC residents (home PUMA)", n_state, len(r))
assert len(r) == 326_729

r["weight"] = pd.to_numeric(r.PWGTP, errors="raise").astype("Int64")
print(f"Weights sum to {r.weight.sum():,} (period-average NYC population; nb 10: 8,480,225)")
assert r.weight.sum() == 8_480_225

r["age"] = pd.to_numeric(r.AGEP, errors="raise").astype("Int64")
before = len(r); r = r[r.age >= 16].copy()
log("keep age 16+ (working age; the universe of the job and pay questions)", before, len(r))

  NY State persons -> NYC residents (home PUMA): 984,474 -> 326,729 rows (-657,745)
Weights sum to 8,480,225 (period-average NYC population; nb 10: 8,480,225)


  keep age 16+ (working age; the universe of the job and pay questions): 326,729 -> 279,311 rows (-47,418)


## 3. Decode and derive (issues 1, 6–8, 10)

- **Blanks** are "not in the question's universe" (no field of degree below a bachelor's, no job code for people who haven't worked in 5 years, no wage for non-workers). They become NULL and are counted below; they are never zeros.
- **Pay in 2024 dollars:** `WAGP × ADJINC / 1,000,000` (one `ADJINC` per response year).
- `degree_level` from `SCHL`: 20 associate's, 21 bachelor's, 22–24 graduate. `enrolled` = attending school now (`SCHG` filled). `works_in_nyc` = place of work in a NYC borough.
- Imputation flags kept for field of degree, occupation and wages (D2 sensitivity runs without imputed rows).

In [4]:
def num(col: str, name: str) -> pd.Series:
    blank = r[col] == ""
    print(f"  {name:<18} blank (not in universe) -> NULL: {blank.sum():>7,} ({blank.mean():.0%})")
    return pd.to_numeric(r[col].mask(blank), errors="raise").astype("Int64")

r["response_year"] = r.SERIALNO.str[:4].astype(int).astype("Int64")
r["is_group_quarters"] = r.SERIALNO.str[4:6].eq("GQ")
r["borough"] = r.PUMA.map(PUMA_BORO)
adj = pd.to_numeric(r.ADJINC, errors="raise") / 1e6
print("ADJINC by year:", r.groupby("response_year").ADJINC.unique().map(lambda a: ", ".join(a)).to_dict())
assert r.groupby("response_year").ADJINC.nunique().eq(1).all()
r["wage_2024usd"] = (num("WAGP", "wage") * adj).round().astype("Int64")
r["earnings_2024usd"] = (num("PERNP", "earnings") * adj).round().astype("Int64")
r["usual_hours"] = num("WKHP", "usual_hours")
r["weeks_worked"] = num("WKWN", "weeks_worked")

schl = num("SCHL", "education_code")
r["education_code"] = r.SCHL.replace("", None)
r["degree_level"] = pd.cut(schl.astype("float64"), bins=[0, 19, 20, 21, 24], labels=["below associate", "associate", "bachelor", "graduate"]).astype("string")
r["enrolled"] = r.SCHG.ne("")
r["works_in_nyc"] = r.POWSP.eq("036") & r.POWPUMA.isin(NYC_POW)
for c in ["FOD1P", "FOD2P", "ESR", "COW", "OCCP", "SOCP", "NAICSP"]:
    print(f"  {c:<18} blank (not in universe) -> NULL: {(r[c] == '').sum():>7,}")
    r[c] = r[c].replace("", None)
assert r.FOD1P.notna().eq(schl >= 21).all(), "field of degree outside bachelor's+"
for c, dst in [("FFODP", "fod_imputed"), ("FOCCP", "occ_imputed"), ("FWAGP", "wage_imputed")]:
    assert set(r[c]) <= {"0", "1", ""}, set(r[c])
    r[dst] = r[c].eq("1")
for c in REP:
    r[c.lower()] = pd.to_numeric(r[c], errors="raise").astype("Int64")
display(pd.crosstab(r.degree_level, r.response_year, values=r.weight, aggfunc="sum").map("{:,.0f}".format))

ADJINC by year: {np.int64(2020): '1222017', np.int64(2021): '1193241', np.int64(2022): '1117193', np.int64(2023): '1049470', np.int64(2024): '1015250'}
  wage               blank (not in universe) -> NULL:       0 (0%)


  earnings           blank (not in universe) -> NULL:       0 (0%)
  usual_hours        blank (not in universe) -> NULL: 107,258 (38%)


  weeks_worked       blank (not in universe) -> NULL: 107,258 (38%)


  education_code     blank (not in universe) -> NULL:       0 (0%)
  FOD1P              blank (not in universe) -> NULL: 169,779
  FOD2P              blank (not in universe) -> NULL: 265,652
  ESR                blank (not in universe) -> NULL:       0
  COW                blank (not in universe) -> NULL:  80,514


  OCCP               blank (not in universe) -> NULL:  80,514
  SOCP               blank (not in universe) -> NULL:  80,514
  NAICSP             blank (not in universe) -> NULL:  80,514


response_year,2020,2021,2022,2023,2024
degree_level,,,,,
associate,"93,174","88,303","82,333","82,496","81,727"
bachelor,"317,814","309,886","315,229","321,085","331,207"
below associate,"790,372","767,301","767,511","739,776","773,511"
graduate,"219,149","212,725","212,854","220,572","225,848"


## 4. Tie-out (card §5): graduation-age graduates

22–27-year-olds with an associate's or bachelor's: nb 10 counted 14,073 rows.

In [5]:
g = r[r.age.between(22, 27) & r.degree_level.isin(["associate", "bachelor"])]
print(f"Rows aged 22-27 with an associate's or bachelor's: {len(g):,}"); assert len(g) == 14_073
print("Still enrolled (weighted):", (g.groupby("degree_level").apply(lambda x: (x.weight * x.enrolled).sum() / x.weight.sum(), include_groups=False) * 100).round(0).to_dict())

Rows aged 22-27 with an associate's or bachelor's: 14,073
Still enrolled (weighted): {'associate': 41.0, 'bachelor': 23.0}


## 5. Stage

In [6]:
stg = (r.rename(columns={"SERIALNO": "serialno", "SPORDER": "sporder", "PUMA": "puma", "SEX": "sex", "FOD1P": "fod1",
                         "FOD2P": "fod2", "ESR": "employment_status", "COW": "class_of_worker", "OCCP": "occp",
                         "SOCP": "socp", "NAICSP": "naicsp"})
       [["serialno", "sporder", "response_year", "puma", "borough", "is_group_quarters", "age", "sex", "education_code",
         "degree_level", "enrolled", "fod1", "fod2", "employment_status", "class_of_worker", "occp", "socp", "naicsp",
         "wage_2024usd", "earnings_2024usd", "usual_hours", "weeks_worked", "works_in_nyc", "fod_imputed", "occ_imputed",
         "wage_imputed", "weight", *[c.lower() for c in REP]]]
       .assign(source_file=F[F.file.str.endswith("csv_pny_2020-2024_5yr.zip")].file.iloc[0])
       .sort_values(["serialno", "sporder"]).reset_index(drop=True))
summary = stage(stg, "stg_census__acs_pums_person", ["serialno", "sporder"], expected_rows=279_311, source_key="acs_pums",
                notebook=NB, conn=conn,
                comment="ACS PUMS 2020-2024 5-year, NYC residents 16+. Weights = period average. Pay in 2024 $. MOE from pwgtp1-80 (SDR).")
display(summary)
if conn is not None:
    conn.close()

  stg_census__acs_pums_person: 279,311 rows x 108 cols, key ['serialno', 'sporder'] unique | local data/interim/staging/stg_census__acs_pums_person.parquet | DB: 279311


,table,rows,cols,key,source_key,notebook,in_db,loaded_utc
0,stg_census__acs_pums_person,279311,108,serialno + sporder,acs_pums,27-load-acs-pums.ipynb,279311,2026-10-14T21:22:26+00:00
